# 🏆 Simulación Predictiva del Mundial 2026
**Modelo basado en Random Forest, Regresión de Poisson y Simulaciones de Monte Carlo**

Este proyecto utiliza datos históricos de partidos, estadísticas de rendimiento (xG, posesión, tiros a puerta) y valor de mercado para predecir el resultado del Mundial 2026.

## 1. Configuración del Entorno y Librerías
Importamos las herramientas necesarias para la manipulación de datos (`pandas`, `numpy`), el modelado predictivo (`scikit-learn`) y conectamos el entorno con Google Drive para acceder a las bases de datos.

In [ ]:
#==========================================================
# BLOQUE 1
# MONTAJE DE GOOGLE DRIVE Y LIBRERÍAS
#==========================================================

from google.colab import drive
drive.mount('/content/drive')
#==========================================================
# LIBRERÍAS
#==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

Mounted at /content/drive


## 2. Carga de Datos
Ingestamos los diferentes datasets que componen nuestro universo de información: datos históricos de partidos, estadísticas consolidadas por equipo, grupos del mundial y rankings de la FIFA.

In [ ]:
#==========================================================
# BLOQUE 2
# CARGAR ARCHIVOS
#==========================================================

base_path = "/content/drive/MyDrive/Simulaciones_Mundial/Data"

datos_mundial = pd.read_csv(f"{base_path}/datos_mundial.csv")

datos_historicos = pd.read_csv(f"{base_path}/datos_historicos.csv")

partidos = pd.read_csv(f"{base_path}/partidos.csv")

grupos = pd.read_csv(f"{base_path}/Grupos_Mundial.csv", sep=';')

ranking = pd.read_csv(f"{base_path}/ranking_fifa.csv")

transfermarkt = pd.read_csv(f"{base_path}/transfermarkt.csv")
print("datos_mundial:", datos_mundial.shape)
print("datos_historicos:", datos_historicos.shape)
print("partidos:", partidos.shape)
print("grupos:", grupos.shape)
print("ranking:", ranking.shape)
print("transfermarkt:", transfermarkt.shape)

print(datos_mundial.columns.tolist()[:20])

print()

print(datos_historicos.columns.tolist()[:20])

datos_mundial: (48, 47)
datos_historicos: (1396, 45)
partidos: (3316, 2778)
grupos: (48, 2)
ranking: (6931, 3)
transfermarkt: (217, 2)
['Equipo', 'Puntos', 'Peso', 'Valor_Mercado_Millones_Eur', 'Tipo_Equipo', 'Fecha', 'Resultado_1X2', 'avg_Goles_esperados_(xG)_total', 'avg_Posesión_total', 'avg_Remates_a_puerta_total', 'avg_Córneres_total', 'avg_Tarjetas_amarillas_total', 'avg_Faltas_total', 'avg_Paradas_total', 'avg_Pases_Pct_total', 'avg_Pases_Exitosos_total', 'avg_Goles_total', 'avg_Ptos_total', 'avg_xG_Share_total', 'avg_Remates_Puerta_Share_total']

['Fecha', 'Equipo_Local', 'Equipo_Visitante', 'Goles_Local', 'Goles_Visitante', 'Peso_Local', 'avg_Goles_esperados_(xG)_total_Local', 'avg_Tarjetas_amarillas_total_Local', 'avg_Faltas_total_Local', 'avg_Remates_a_puerta_3_Local', 'avg_Córneres_3_Local', 'avg_Tarjetas_amarillas_3_Local', 'avg_Faltas_3_Local', 'avg_Paradas_3_Local', 'avg_Pases_Pct_3_Local', 'avg_Pases_Exitosos_3_Local', 'avg_Paradas_Share_3_Local', 'Peso_Visitante', 'Res

In [ ]:
# Elimina espacios en los nombres de las columnas
datos_mundial.columns = datos_mundial.columns.str.strip()
grupos.columns = grupos.columns.str.strip()
datos_historicos.columns = datos_historicos.columns.str.strip()
partidos.columns = partidos.columns.str.strip()

## 3. Normalización y Limpieza de Datos
Para que el modelo pueda cruzar la información correctamente, es vital estandarizar los nombres de los países. Aplicamos un diccionario de mapeo para corregir variaciones en los nombres (ej. "USA" -> "Estados Unidos").

In [ ]:
#==========================================================
# BLOQUE 3
# NORMALIZACIÓN DE NOMBRES DE EQUIPOS
#==========================================================

# Diccionario de equivalencias
normalizar_equipo = {

    # América
    "USA": "Estados Unidos",
    "United States": "Estados Unidos",

    # Europa
    "England": "Inglaterra",
    "Netherlands": "Países Bajos",
    "Holland": "Países Bajos",
    "Switzerland": "Suiza",
    "Belgium": "Bélgica",
    "Germany": "Alemania",
    "Spain": "España",
    "Portugal": "Portugal",
    "France": "Francia",
    "Austria": "Austria",
    "Sweden": "Suecia",
    "Norway": "Noruega",
    "Bosnia & Herzegovina": "Bosnia y Herzegovina",
    "Bosnia and Herzegovina": "Bosnia y Herzegovina",

    # África
    "Ivory Coast": "Costa de Marfil",
    "South Africa": "Sudáfrica",
    "Cape Verde": "Cabo Verde",
    "Egypt": "Egipto",
    "Ghana": "Ghana",
    "Morocco": "Marruecos",

    # Asia
    "South Korea": "Corea del Sur",
    "Korea Republic": "Corea del Sur",
    "Japan": "Japón",
    "Iran": "Irán",

    # Oceanía
    "Australia": "Australia",

    # Sudamérica
    "Brazil": "Brasil",
    "Argentina": "Argentina",
    "Colombia": "Colombia",
    "Paraguay": "Paraguay",
    "Ecuador": "Ecuador",

    # Norteamérica
    "Canada": "Canadá",
    "Mexico": "México",
    "Senegal": "Senegal"
}

def normalizar_nombre(nombre):
    """
    Devuelve el nombre normalizado de un equipo.
    Si no existe en el diccionario, devuelve el original.
    """
    if pd.isna(nombre):
        return nombre

    nombre = str(nombre).strip()

    return normalizar_equipo.get(nombre, nombre)

    # datos_mundial
datos_mundial["Equipo"] = datos_mundial["Equipo"].apply(normalizar_nombre)

# históricos
datos_historicos["Equipo_Local"] = datos_historicos["Equipo_Local"].apply(normalizar_nombre)

datos_historicos["Equipo_Visitante"] = datos_historicos["Equipo_Visitante"].apply(normalizar_nombre)

# grupos
grupos['Equipo'] = grupos['Equipo'].apply(normalizar_nombre)

In [ ]:
print('Las columnas actuales del DataFrame `grupos` son:')
print(grupos.columns.tolist())

Las columnas actuales del DataFrame `grupos` son:
['Equipo', 'Grupo']


## 4. Ingeniería de Características (Feature Engineering)
Separamos las variables predictoras (features) de los equipos locales y visitantes. Nuestro objetivo (target) será predecir los `Goles_Local` y `Goles_Visitante` basándonos en métricas de rendimiento acumuladas.

In [ ]:
#==========================================================
# BLOQUE 4
# CONSTRUCCIÓN AUTOMÁTICA DE FEATURES
#==========================================================

# Variables Local
features_local = [
    c for c in datos_historicos.columns
    if c.startswith("avg_") and c.endswith("_Local")
]

# Variables Visitante
features_visit = [
    c for c in datos_historicos.columns
    if c.startswith("avg_") and c.endswith("_Visitante")
]

print("Variables Local:", len(features_local))
print("Variables Visitante:", len(features_visit))

# Unimos todas las disponibles
X = datos_historicos[features_local + features_visit].copy()

# Targets
y_local = datos_historicos["Goles_Local"]
y_visitante = datos_historicos["Goles_Visitante"]

print("\nDimensiones de X:", X.shape)

Variables Local: 11
Variables Visitante: 13

Dimensiones de X: (1396, 24)


In [ ]:
#==========================================================
# LIMPIEZA
#==========================================================

# Eliminar partidos sin goles registrados
datos_validos = datos_historicos.dropna(
    subset=["Goles_Local", "Goles_Visitante"]
).copy()

# Reconstruir X e y con los datos válidos
X = datos_validos[features_local + features_visit]

y_local = datos_validos["Goles_Local"]
y_visitante = datos_validos["Goles_Visitante"]

# Rellenar valores faltantes con la media
X = X.fillna(X.mean(numeric_only=True))

print("X:", X.shape)
print("Nulos:", X.isnull().sum().sum())

X: (1394, 24)
Nulos: 0


## 5. Preparación del Modelo (Train/Test Split)
Dividimos nuestro conjunto de datos históricos: 80% para que el modelo aprenda y 20% para evaluar su rendimiento. Además, aplicamos `StandardScaler` para que todas las variables numéricas tengan el mismo peso estadístico.

In [ ]:
#==========================================================
# BLOQUE 5
# DIVISIÓN DEL CONJUNTO DE DATOS Y ESCALADO
#==========================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# División entrenamiento/prueba
X_train, X_test, y_train_local, y_test_local = train_test_split(
    X,
    y_local,
    test_size=0.20,
    random_state=42
)

_, _, y_train_visit, y_test_visit = train_test_split(
    X,
    y_visitante,
    test_size=0.20,
    random_state=42
)

# Inicializar y ajustar el StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)
print("✅ StandardScaler inicializado y ajustado.")

Entrenamiento: (1115, 24)
Prueba: (279, 24)
✅ StandardScaler inicializado y ajustado.


In [ ]:
#==========================================================
# RANDOM FOREST
#==========================================================

from sklearn.ensemble import RandomForestRegressor

rf_local = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_visit = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_local.fit(X_train, y_train_local)
rf_visit.fit(X_train, y_train_visit)

print("✅ Modelos entrenados correctamente.")

✅ Modelos entrenados correctamente.


In [ ]:
pred_local = rf_local.predict(X_test)
pred_visit = rf_visit.predict(X_test)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("========== LOCAL ==========")

print("MAE :", mean_absolute_error(y_test_local, pred_local))
print("RMSE:", np.sqrt(mean_squared_error(y_test_local, pred_local)))
print("R²  :", r2_score(y_test_local, pred_local))

print()

print("======= VISITANTE ========")

print("MAE :", mean_absolute_error(y_test_visit, pred_visit))
print("RMSE:", np.sqrt(mean_squared_error(y_test_visit, pred_visit)))
print("R²  :", r2_score(y_test_visit, pred_visit))

========== LOCAL ==========
MAE : 1.0121492085937913
RMSE: 1.3966400233903893
R²  : 0.16735078061591224

======= VISITANTE ========
MAE : 0.942234163373392
RMSE: 1.2608378104238158
R²  : 0.2988885394640719


## 6. Entrenamiento del Algoritmo (Random Forest Regressor)
Entrenamos dos modelos independientes: uno para predecir el comportamiento ofensivo del equipo local y otro para el visitante. Utilizamos un Bosque Aleatorio con 200 árboles (`n_estimators=200`) y una profundidad máxima controlada (`max_depth=10`) para evitar el sobreajuste. Al final, evaluamos su error (MAE, RMSE) y capacidad de varianza (R²).

In [ ]:
#==========================================================
# BLOQUE 6
# FUNCIÓN PRINCIPAL DE PREDICCIÓN
#==========================================================
import pandas as pd
import random

def predecir_partido(equipo_local, equipo_visitante, fase_eliminatoria=False):
    """
    Predice el resultado entre dos equipos.
    Si fase_eliminatoria=True, resuelve los empates simulando penales o tiempo extra.
    """
    # 1. Normalizar nombres
    local_norm = normalizar_nombre(equipo_local)
    visit_norm = normalizar_nombre(equipo_visitante)

    # 2. Buscar estadísticas en datos_mundial
    try:
        data_local = datos_mundial[datos_mundial['Equipo'] == local_norm].iloc[0]
        data_visit = datos_mundial[datos_mundial['Equipo'] == visit_norm].iloc[0]
    except IndexError:
        raise ValueError(f"Error: Al menos uno de los equipos ('{local_norm}' o '{visit_norm}') no existe en datos_mundial.")

    # 3. Construir vector de entrada
    X_pred_dict = {}

    # Extraer variables del local
    for col in features_local:
        base_col = col.replace('_Local', '')
        X_pred_dict[col] = data_local.get(base_col, 0)

    # Extraer variables del visitante
    for col in features_visit:
        base_col = col.replace('_Visitante', '')
        X_pred_dict[col] = data_visit.get(base_col, 0)

    # Crear DataFrame asegurando el orden exacto de las columnas
    X_pred = pd.DataFrame([X_pred_dict])[X_train.columns]

    # Limpiar posibles valores faltantes
    X_pred = X_pred.fillna(X_pred.mean(numeric_only=True).fillna(0))

    # 4. Predecir goles (sin scaler, ya que rf se entrenó con X_train puro)
    goles_local_pred = rf_local.predict(X_pred)[0]
    goles_visit_pred = rf_visit.predict(X_pred)[0]

    # Redondeo para obtener goles enteros en el marcador final
    goles_local = int(round(goles_local_pred))
    goles_visit = int(round(goles_visit_pred))

    # 5. Determinar ganador y resolver empates
    if goles_local > goles_visit:
        ganador = local_norm
    elif goles_visit > goles_local:
        ganador = visit_norm
    else:
        if fase_eliminatoria:
            # Regla de desempate usando los decimales de la predicción como ventaja
            if goles_local_pred > goles_visit_pred:
                ganador = local_norm
            elif goles_visit_pred > goles_local_pred:
                ganador = visit_norm
            else:
                # Desempate total al azar (simulando tanda de penales impredecible)
                ganador = random.choice([local_norm, visit_norm])
        else:
            ganador = "Empate"

    # Mostrar resultados en consola
    print(f"Predicción: {local_norm} vs {visit_norm}")
    print(f"Marcador: {goles_local} - {goles_visit}")
    print(f"Ganador: {ganador}")
    if fase_eliminatoria and goles_local == goles_visit:
        print("(Victoria decidida en desempate/penales)")
    print("-" * 30)

    return {
        "Local": local_norm,
        "Visitante": visit_norm,
        "Goles_Local": goles_local,
        "Goles_Visitante": goles_visit,
        "Ganador": ganador
    }

# Prueba de la función
resultado = predecir_partido("Brasil", "España", fase_eliminatoria=True)

Predicción: Brasil vs España
Marcador: 1 - 2
Ganador: España
------------------------------


## 7. Motor de Simulación Probabilística
El Random Forest nos da la esperanza matemática (los goles esperados o $\lambda$). Sin embargo, el fútbol tiene un componente de azar. Para simular la realidad, pasamos esa predicción por una **Distribución de Poisson**, lo que nos permite generar resultados discretos y resolver empates en fases eliminatorias.

In [ ]:
#==========================================================
# BLOQUE 7 COMPLETO
# MOTOR DE SIMULACIÓN DE FASES ELIMINATORIAS (CON POISSON)
#==========================================================
import numpy as np
import pandas as pd

def predecir_partido_probabilistico(equipo_local, equipo_visitante, verbose=True):
    local_norm = normalizar_nombre(equipo_local)
    visit_norm = normalizar_nombre(equipo_visitante)

    # Validación de existencia de equipos
    if not (datos_mundial['Equipo'] == local_norm).any():
        raise ValueError(f"❌ Error: El equipo '{local_norm}' no existe en datos_mundial.")
    if not (datos_mundial['Equipo'] == visit_norm).any():
        raise ValueError(f"❌ Error: El equipo '{visit_norm}' no existe en datos_mundial.")

    data_local = datos_mundial[datos_mundial['Equipo'] == local_norm].iloc[0]
    data_visit = datos_mundial[datos_mundial['Equipo'] == visit_norm].iloc[0]

    X_pred_dict = {}
    for col in features_local:
        X_pred_dict[col] = data_local.get(col.replace('_Local', ''), 0)
    for col in features_visit:
        X_pred_dict[col] = data_visit.get(col.replace('_Visitante', ''), 0)

    X_pred = pd.DataFrame([X_pred_dict])[X_train.columns]
    X_pred = X_pred.fillna(X_pred.mean(numeric_only=True).fillna(0))

    lambda_local = max(0.1, rf_local.predict(X_pred)[0])
    lambda_visit = max(0.1, rf_visit.predict(X_pred)[0])

    goles_local = np.random.poisson(lambda_local)
    goles_visit = np.random.poisson(lambda_visit)

    if goles_local > goles_visit:
        ganador = local_norm
    elif goles_visit > goles_local:
        ganador = visit_norm
    else:
        ganador = np.random.choice([local_norm, visit_norm])

    if verbose:
        print(f"{local_norm} ({goles_local}) vs ({goles_visit}) {visit_norm} -> Avanza: {ganador}")

    return ganador

def simular_ronda(equipos, nombre_fase, verbose=True):
    ganadores = []
    if verbose:
        print(f"\n--- {nombre_fase.upper()} ---")
    for i in range(0, len(equipos), 2):
        ganador = predecir_partido_probabilistico(equipos[i], equipos[i+1], verbose)
        ganadores.append(ganador)
    return ganadores

def simular_fase_final(equipos_ronda_inicial, verbose=True):
    """
    Simula las 5 rondas necesarias para un bracket de 32 equipos.
    """
    dieciseisavos = simular_ronda(equipos_ronda_inicial, "Dieciseisavos de Final", verbose)
    octavos = simular_ronda(dieciseisavos, "Octavos de Final", verbose)
    cuartos = simular_ronda(octavos, "Cuartos de Final", verbose)
    semis = simular_ronda(cuartos, "Semifinales", verbose)
    campeon = simular_ronda(semis, "Gran Final", verbose)

    if verbose:
        print(f"\n🏆 ¡EL CAMPEÓN ES {campeon[0].upper()}! 🏆\n")

    return campeon[0]

# ==========================================
# ORDEN DE EJECUCIÓN CON TU CALENDARIO (32 EQUIPOS)
# ==========================================

# Aquí están los cruces basados en tu imagen.
# Los que decían "Third Place" o faltaban fueron rellenados para completar los 32.
equipos_dieciseisavos = [
    # Cruces confirmados en tu imagen
    "Sudáfrica", "Canadá",
    "Brasil", "Japón",
    "Alemania", "Paraguay",
    "Países Bajos", "Marruecos",
    "Costa de Marfil", "Noruega",
    "Francia", "Suecia",

    # Cruces con placeholders en tu imagen (rellenados con tu lista de 48)
    "México", "República Checa",         # Sustituye a "Third Place Group C/E/F/H/I"
    "Argentina", "Corea del Sur",        # Sustituye a "Group L Winner vs Third Place..."
    "Bélgica", "Catar",                  # Sustituye a "Third Place Group A/E/H/I/J"

    # Cruce de la imagen con nombres corregidos para tu base de datos
    "EE. UU.", "Bosnia-Herzegovina",

    # Más cruces con placeholders rellenados
    "España", "Haití",                   # Sustituye a "Group J 2nd Place"
    "Inglaterra", "Escocia",             # Sustituye a "Group K 2nd Place vs Group L 2nd Place"
    "Suiza", "Curazao",                  # Sustituye a "Third Place Group E/F/G/I/J"

    # Los 3 partidos faltantes para llegar a 16 enfrentamientos (32 equipos)
    "Portugal", "Turquía",
    "Colombia", "Australia",
    "Uruguay", "Ecuador"
]

print("Iniciando simulación de Dieciseisavos (32 equipos) con el calendario oficial...")
campeon_simulacion = simular_fase_final(equipos_dieciseisavos, verbose=True)

Iniciando simulación de Dieciseisavos (32 equipos) con el calendario oficial...

--- DIECISEISAVOS DE FINAL ---
Sudáfrica (2) vs (0) Canadá -> Avanza: Sudáfrica
Brasil (2) vs (0) Japón -> Avanza: Brasil
Alemania (0) vs (2) Paraguay -> Avanza: Paraguay
Países Bajos (1) vs (1) Marruecos -> Avanza: Marruecos
Costa de Marfil (2) vs (0) Noruega -> Avanza: Costa de Marfil
Francia (3) vs (0) Suecia -> Avanza: Francia
México (2) vs (1) República Checa -> Avanza: México
Argentina (0) vs (1) Corea del Sur -> Avanza: Corea del Sur
Bélgica (1) vs (1) Catar -> Avanza: Bélgica
EE. UU. (1) vs (4) Bosnia-Herzegovina -> Avanza: Bosnia-Herzegovina
España (1) vs (0) Haití -> Avanza: España
Inglaterra (5) vs (0) Escocia -> Avanza: Inglaterra
Suiza (2) vs (0) Curazao -> Avanza: Suiza
Portugal (1) vs (0) Turquía -> Avanza: Portugal
Colombia (2) vs (1) Australia -> Avanza: Colombia
Uruguay (3) vs (0) Ecuador -> Avanza: Uruguay

--- OCTAVOS DE FINAL ---
Sudáfrica (3) vs (3) Brasil -> Avanza: Sudáfrica
Paragua

## 8. Simulación de Monte Carlo
Ejecutamos el motor de simulación 1,000 veces consecutivas. Al registrar qué equipo gana el torneo en cada iteración, podemos calcular la probabilidad real que tiene cada selección de llevarse la copa. Al finalizar, exportamos las probabilidades a un archivo `.csv`.

In [ ]:
#==========================================================
# BLOQUE 8
# GUARDAR PREDICCIONES Y EXPORTAR CSV
#==========================================================
from google.colab import files

# --- Configuración de la Simulación Monte Carlo ---
n_simulaciones = 1000 # Número de veces que se simulará el torneo

print(f"Iniciando {n_simulaciones} simulaciones de Monte Carlo...")

# Lista para guardar los campeones de cada simulación
campeones_simulados = []

for i in range(n_simulaciones):
    # Ejecutar la simulación completa sin imprimir todos los partidos
    campeon = simular_fase_final(equipos_dieciseisavos, verbose=False)
    campeones_simulados.append(campeon)

print("✅ Simulaciones completadas.")

# Contar la frecuencia de cada campeón
resultados_monte_carlo = pd.Series(campeones_simulados).value_counts().reset_index()
resultados_monte_carlo.columns = ['Equipo', 'Veces_Campeon']

# Opcional: Calcular el porcentaje de veces que cada equipo ganó
resultados_monte_carlo['Probabilidad_Campeon_%'] = (resultados_monte_carlo['Veces_Campeon'] / n_simulaciones) * 100
resultados_monte_carlo = resultados_monte_carlo.sort_values(by='Probabilidad_Campeon_%', ascending=False)

print("\n--- Top 5 posibles campeones ---")
print(resultados_monte_carlo.head())

# Asegurarnos de que el nombre del archivo sea descriptivo
nombre_archivo = "predicciones_monte_carlo_mundial.csv"

# 1. Guardar el archivo en la misma ruta de Drive que definiste en el Bloque 2
ruta_guardado = f"{base_path}/{nombre_archivo}"
resultados_monte_carlo.to_csv(ruta_guardado, index=False, encoding='utf-8')
print(f"✅ Resultados guardados en Drive: {ruta_guardado}")

# 2. Descargar el archivo directamente a tu computadora
try:
    files.download(ruta_guardado)
    print("⬇️ Descarga iniciada correctamente. Ya puedes enviarlo.")
except Exception as e:
    print(f"⚠️ Hubo un problema al intentar descargar el archivo automáticamente: {e}")
    print("Puedes descargarlo manualmente desde el panel de archivos a la izquierda.")

Iniciando 1000 simulaciones de Monte Carlo...
✅ Simulaciones completadas.

--- Top 5 posibles campeones ---
            Equipo  Veces_Campeon  Probabilidad_Campeon_%
0         Alemania            129                    12.9
1           España            115                    11.5
2  Costa de Marfil            104                    10.4
3        Sudáfrica             89                     8.9
4         Portugal             85                     8.5
✅ Resultados guardados en Drive: /content/drive/MyDrive/Simulaciones_Mundial/Data/predicciones_monte_carlo_mundial.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ Descarga iniciada correctamente. Ya puedes enviarlo.


## 9. Simulación Individual y Exportación Detallada del Cuadro Final
Finalmente, realizamos una simulación del torneo guardando el detalle exacto de cada partido: quién contra quién, los goles esperados (xG) de ambos, el marcador final simulado y cómo se resolvió (tiempo regular o penales). Esto se exporta como un `.csv` para su posterior análisis o visualización.

In [ ]:
#==========================================================
# BLOQUE 9
# SIMULACIÓN INDIVIDUAL CON REGISTRO DETALLADO Y DESCARGA CSV
#==========================================================
import numpy as np
import pandas as pd
from google.colab import files

def predecir_partido_registro(equipo_local, equipo_visitante, fase, registro, verbose=True):
    local_norm = normalizar_nombre(equipo_local)
    visit_norm = normalizar_nombre(equipo_visitante)

    # Validaciones
    if not (datos_mundial['Equipo'] == local_norm).any():
        raise ValueError(f"❌ Error: El equipo '{local_norm}' no existe en datos_mundial.")
    if not (datos_mundial['Equipo'] == visit_norm).any():
        raise ValueError(f"❌ Error: El equipo '{visit_norm}' no existe en datos_mundial.")

    data_local = datos_mundial[datos_mundial['Equipo'] == local_norm].iloc[0]
    data_visit = datos_mundial[datos_mundial['Equipo'] == visit_norm].iloc[0]

    X_pred_dict = {}
    for col in features_local:
        X_pred_dict[col] = data_local.get(col.replace('_Local', ''), 0)
    for col in features_visit:
        X_pred_dict[col] = data_visit.get(col.replace('_Visitante', ''), 0)

    X_pred = pd.DataFrame([X_pred_dict])[X_train.columns]
    X_pred = X_pred.fillna(X_pred.mean(numeric_only=True).fillna(0))

    # Predicción xG (Esperanza matemática)
    lambda_local = max(0.1, rf_local.predict(X_pred)[0])
    lambda_visit = max(0.1, rf_visit.predict(X_pred)[0])

    # Simulación Poisson
    goles_local = np.random.poisson(lambda_local)
    goles_visit = np.random.poisson(lambda_visit)

    detalle = "Tiempo regular"
    if goles_local > goles_visit:
        ganador = local_norm
    elif goles_visit > goles_local:
        ganador = visit_norm
    else:
        ganador = np.random.choice([local_norm, visit_norm])
        detalle = "Empate: avanza por penales"

    # Guardar en el registro
    registro.append({
        "Fase": fase,
        "Local": local_norm,
        "Visitante": visit_norm,
        "Marcador_Predicho": f"{goles_local}-{goles_visit}",
        "Avanza": ganador,
        "xG_L": round(lambda_local, 2),
        "xG_V": round(lambda_visit, 2),
        "Detalle": detalle
    })

    if verbose:
        print(f"{local_norm} ({goles_local}) vs ({goles_visit}) {visit_norm} -> Avanza: {ganador}")

    return ganador

def simular_ronda_registro(equipos, nombre_fase, registro, verbose=True):
    ganadores = []
    if verbose:
        print(f"\n--- {nombre_fase.upper()} ---")
    for i in range(0, len(equipos), 2):
        ganador = predecir_partido_registro(equipos[i], equipos[i+1], nombre_fase, registro, verbose)
        ganadores.append(ganador)
    return ganadores

def simular_fase_final_exportable(equipos_ronda_inicial, verbose=True):
    registro_partidos = []

    dieciseisavos = simular_ronda_registro(equipos_ronda_inicial, "Dieciseisavos de Final", registro_partidos, verbose)
    octavos = simular_ronda_registro(dieciseisavos, "Octavos de Final", registro_partidos, verbose)
    cuartos = simular_ronda_registro(octavos, "Cuartos de Final", registro_partidos, verbose)
    semis = simular_ronda_registro(cuartos, "Semifinales", registro_partidos, verbose)
    campeon = simular_ronda_registro(semis, "Gran Final", registro_partidos, verbose)

    if verbose:
        print(f"\n🏆 ¡EL CAMPEÓN ES {campeon[0].upper()}! 🏆\n")

    # Convertir el registro en un DataFrame de Pandas
    df_registro = pd.DataFrame(registro_partidos)
    return campeon[0], df_registro

# ==========================================
# DEFINICIÓN DE EQUIPOS Y EJECUCIÓN
# ==========================================

# Incluimos la lista aquí para que no dependa de celdas anteriores
equipos_dieciseisavos = [
    "Sudáfrica", "Canadá",
    "Brasil", "Japón",
    "Alemania", "Paraguay",
    "Países Bajos", "Marruecos",
    "Costa de Marfil", "Noruega",
    "Francia", "Suecia",
    "México", "República Checa",
    "Argentina", "Corea del Sur",
    "Bélgica", "Catar",
    "EE. UU.", "Bosnia-Herzegovina",
    "España", "Haití",
    "Inglaterra", "Escocia",
    "Suiza", "Curazao",
    "Portugal", "Turquía",
    "Colombia", "Australia",
    "Uruguay", "Ecuador"
]

print("Ejecutando simulación para exportar resultados...")
campeon_unico, df_resultados_torneo = simular_fase_final_exportable(equipos_dieciseisavos, verbose=False)

# Mostrar una vista previa de los datos
print("\nVista previa del archivo a exportar:")
display(df_resultados_torneo.head(10))

# Guardar y descargar
nombre_archivo_torneo = "detalle_simulacion_torneo.csv"
ruta_torneo = f"{base_path}/{nombre_archivo_torneo}"

df_resultados_torneo.to_csv(ruta_torneo, index=False, encoding='utf-8')
print(f"\n✅ Archivo guardado en: {ruta_torneo}")

try:
    files.download(ruta_torneo)
    print("⬇️ Iniciando descarga del archivo CSV con el detalle de los partidos...")
except Exception as e:
    print(f"⚠️ No se pudo descargar automáticamente: {e}")

Ejecutando simulación para exportar resultados...

Vista previa del archivo a exportar:


,Fase,Local,Visitante,Marcador_Predicho,Avanza,xG_L,xG_V,Detalle
0,Dieciseisavos de Final,Sudáfrica,Canadá,1-0,Sudáfrica,2.08,0.87,Tiempo regular
1,Dieciseisavos de Final,Brasil,Japón,1-2,Japón,1.44,1.15,Tiempo regular
2,Dieciseisavos de Final,Alemania,Paraguay,7-1,Alemania,2.36,0.58,Tiempo regular
3,Dieciseisavos de Final,Países Bajos,Marruecos,1-0,Países Bajos,1.53,0.90,Tiempo regular
4,Dieciseisavos de Final,Costa de Marfil,Noruega,4-1,Costa de Marfil,2.06,1.01,Tiempo regular
5,Dieciseisavos de Final,Francia,Suecia,1-0,Francia,1.43,0.81,Tiempo regular
6,Dieciseisavos de Final,México,República Checa,3-0,México,1.69,0.80,Tiempo regular
7,Dieciseisavos de Final,Argentina,Corea del Sur,1-1,Corea del Sur,1.63,0.92,Empate: avanza por penales
8,Dieciseisavos de Final,Bélgica,Catar,1-1,Catar,2.22,0.83,Empate: avanza por penales
9,Dieciseisavos de Final,EE. UU.,Bosnia-Herzegovina,2-2,EE. UU.,1.19,0.96,Empate: avanza por penales



✅ Archivo guardado en: /content/drive/MyDrive/Simulaciones_Mundial/Data/detalle_simulacion_torneo.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ Iniciando descarga del archivo CSV con el detalle de los partidos...


In [1]:
"""
01_Scraping/obtener_datos_deportivos.py
---------------------------------------
Este script se encarga de extraer estadísticas en tiempo real
de fuentes deportivas para alimentar el modelo de simulación.
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def extraer_datos_web():
    print("Iniciando proceso de extracción...")
    # Aquí irá tu lógica de scraping con BeautifulSoup o Selenium
    # Por ahora, simulamos la obtención de datos
    data = {"equipo": ["Brasil", "Alemania"], "goles_esperados": [2.1, 1.9]}
    df = pd.DataFrame(data)

    # Guardamos el resultado en la carpeta Data/
    df.to_csv("../Data/nuevos_datos_scraping.csv", index=False)
    print("Datos extraídos y guardados exitosamente.")

if __name__ == "__main__":
    extraer_datos_web()

Iniciando proceso de extracción...


OSError: Cannot save file into a non-existent directory: '../Data'